# ML — treinamento e avaliação

Parte do bundle Financial Services Medallion. Dados sintéticos.
DATASET_ID e CATALOG_NAME devem ser iguais em todas as tarefas.
Cada tarefa lê suas entradas das tabelas Delta persistidas, sem depender de variáveis de outra sessão.
Destinos existentes provocam falha para evitar sobrescrita; use novo DATASET_ID em uma nova carga.
Rótulos aleatórios: métricas de ML demonstrativas.


In [ ]:
import json, math, random, re
from datetime import datetime, timedelta
from decimal import Decimal
from uuid import uuid4
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DecimalType, IntegerType

if 'spark' not in globals() or 'oidlUtils' not in globals():
    raise RuntimeError('Execute no runtime AIDP com compute anexado.')

def parameter(name, default):
    value = oidlUtils.parameters.getParameter(name, str(default))
    return str(default) if value is None else str(value).strip()

def ident(s):
    if not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*', s):
        raise ValueError(f'Identificador inválido: {s!r}')
    return f'`{s}`'

catalog = parameter('CATALOG_NAME', 'default')
accounts = int(parameter('ACCOUNT_COUNT', 5000))
seed = int(parameter('SEED', 42))
num_trees = int(parameter('RF_NUM_TREES', 100))
max_depth = int(parameter('RF_MAX_DEPTH', 10))
liquid = parameter('ENABLE_LIQUID_CLUSTERING', 'false').lower() == 'true'
if not 100 <= accounts <= 5000 or not 1 <= num_trees <= 100 or not 1 <= max_depth <= 10:
    raise ValueError('Limites do demo: 100–5000 contas, 1–100 árvores, profundidade 1–10.')
run_id = parameter('DATASET_ID', 'bundle_dev')
if not re.fullmatch(r'[A-Za-z][A-Za-z0-9_]{0,47}', run_id):
    raise ValueError('DATASET_ID deve começar com letra e ter até 48 letras, números ou _.')
schemas = {layer: f'finance_{layer}_{run_id}' for layer in ('bronze', 'silver', 'gold')}
def table(layer, name):
    return '.'.join(ident(v) for v in (catalog, schemas[layer], name))
bronze_table = table('bronze', 'account_transactions_bronze')
silver_table = table('silver', 'account_transactions_silver')
analytics_table = table('gold', 'transaction_analytics_gold')
features_table = table('gold', 'fraud_detection_model_gold')
result = {'run_id': run_id, 'catalog': catalog, 'schemas': schemas, 'accounts': accounts, 'seed': seed, 'liquid_clustering_enabled': liquid}
def write_delta(frame, target, keys):
    if liquid:
        view = 'finance_stage_' + uuid4().hex[:8]
        frame.createOrReplaceTempView(view)
        spark.sql(f'CREATE TABLE {target} USING DELTA CLUSTER BY ({keys}) AS SELECT * FROM {view}')
        spark.catalog.dropTempView(view)
    else:
        frame.write.format('delta').mode('errorifexists').saveAsTable(target)

def ensure_schema(layer):
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {ident(catalog)}.{ident(schemas[layer])}')

def require_table(target):
    if not spark.catalog.tableExists(target):
        raise RuntimeError(f'Execute a camada anterior primeiro: {target}')


In [ ]:
require_table(features_table)
ml_features = spark.table(features_table)
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

ml_input = ml_features.withColumn('amount_ml', F.col('amount').cast('double'))
train_data, test_data = ml_input.randomSplit([0.8, 0.2], seed=seed)
train_data, test_data = train_data.cache(), test_data.cache()
train_count, test_count = train_data.count(), test_data.count()
if train_count == 0 or test_count == 0 or train_data.select('is_fraud').distinct().count() != 2 or test_data.select('is_fraud').distinct().count() != 2:
    raise ValueError('Treino e teste devem ser não vazios e conter ambas as classes.')
indexers = [StringIndexer(inputCol=c, outputCol=c + '_index', handleInvalid='keep') for c in ('transaction_type', 'merchant_category', 'location')]
feature_names = ['amount_ml','month','day_of_week','hour','transaction_type_index','merchant_category_index','location_index']
assembler = VectorAssembler(inputCols=feature_names, outputCol='features')
scaler = StandardScaler(inputCol='features', outputCol='scaled_features')
rf = RandomForestClassifier(labelCol='is_fraud', featuresCol='scaled_features', numTrees=num_trees, maxDepth=max_depth, seed=seed)
pipeline = Pipeline(stages=indexers + [assembler, scaler, rf])
print(f'FINANCE_STAGE=training_started train={train_count} test={test_count} trees={num_trees}')
model = pipeline.fit(train_data)
predictions = model.transform(test_data).cache()
auc = BinaryClassificationEvaluator(labelCol='is_fraud', rawPredictionCol='rawPrediction', metricName='areaUnderROC').evaluate(predictions)
assert math.isfinite(auc) and 0 <= auc <= 1
assert predictions.count() == test_count
confusion = {(int(r.is_fraud), int(r.prediction)): r['count'] for r in predictions.groupBy('is_fraud','prediction').count().collect()}
tp, tn = confusion.get((1,1),0), confusion.get((0,0),0)
fp, fn = confusion.get((0,1),0), confusion.get((1,0),0)
result.update(train_rows=train_count, test_rows=test_count, rf_num_trees=num_trees, rf_max_depth=max_depth, auc=float(auc),
    accuracy=(tp+tn)/test_count, precision=tp/(tp+fp) if tp+fp else 0.0, recall=tp/(tp+fn) if tp+fn else 0.0,
    confusion_matrix={f'actual_{a}_pred_{p}':count for (a,p),count in confusion.items()},
    feature_importance={name:float(value) for name,value in zip(feature_names, model.stages[-1].featureImportances)})
print('FINANCE_STAGE=training_complete ' + json.dumps(result, default=str))
predictions.unpersist()
train_data.unpersist()
test_data.unpersist()


In [ ]:

result.update(stage='train', status='SUCCESS', synthetic_labels_only=True)
result['tables'] = [t.replace('`', '') for t in (features_table,)]
print('AIDP_FINANCE_BUNDLE_STAGE_RESULT=' + json.dumps(result, sort_keys=True))
